# Macroeconomic Data Preparation

## Objective

The objective of this notebook is to collect, clean, and consolidate key macroeconomic indicators into a unified quarterly dataset for use in the IFRS 9 Expected Credit Loss (ECL) framework. These macroeconomic variables will serve as the foundation for developing the Macroeconomic Scenario Engine, enabling forward-looking credit risk assessment and stress testing of the Probability of Default (PD) model.

## Macroeconomic Variables

The master dataset includes the following macroeconomic indicators:

- **Real GDP Growth:** Overall economic activity and business cycle conditions.
- **Unemployment Rate:** Labour market conditions and borrower income stability.
- **Federal Funds Rate:** Monetary policy and borrowing cost environment.
- **Consumer Price Index (CPI):** Inflation and purchasing power.
- **House Price Index (HPI):** Housing market performance and household wealth.
- **Disposable Personal Income (DPI):** Consumer repayment capacity and financial resilience.

## Deliverables

The output of this notebook includes:

- **Macroeconomic Dataset:** A master dataset ready for exploratory analysis, scenario generation, and macroeconomic forecasting.

In [1]:
import pandas as pd
import numpy as np

In [2]:
gdp = pd.read_csv("../data/Real_GDP.csv")
unemp = pd.read_csv("../data/Unemployment_Rate.csv")
fed = pd.read_csv("../data/Federal_Funds_Rate.csv")
cpi = pd.read_csv("../data/Consumer_Price_Index.csv")
hpi = pd.read_csv("../data/House_Price_Index.csv")
dpi = pd.read_csv("../data/Disposable_Personal_Income.csv")


In [3]:
datasets = [gdp, unemp, fed, cpi, hpi, dpi]

for df in datasets:
    df["observation_date"] = pd.to_datetime(df["observation_date"])

In [4]:
gdp.columns = ["observation_date", "Real_GDP"]
unemp.columns = ["observation_date", "Unemployment_Rate"]
fed.columns = ["observation_date", "Federal_Funds_Rate"]
cpi.columns = ["observation_date", "CPI"]
hpi.columns = ["observation_date", "House_Price_Index"]
dpi.columns = ["observation_date", "Disposable_Personal_Income"]

In [5]:
unemp_q = (
    unemp
    .set_index("observation_date")
    .resample("Q")
    .mean()
    .reset_index()
)

fed_q = (
    fed
    .set_index("observation_date")
    .resample("Q")
    .mean()
    .reset_index()
)

cpi_q = (
    cpi
    .set_index("observation_date")
    .resample("Q")
    .mean()
    .reset_index()
)

hpi_q = (
    hpi
    .set_index("observation_date")
    .resample("Q")
    .mean()
    .reset_index()
)

dpi_q = (
    dpi
    .set_index("observation_date")
    .resample("Q")
    .mean()
    .reset_index()
)

C:\Users\Lindsey Wang\AppData\Local\Temp\ipykernel_2312\2221768622.py:2: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  unemp
C:\Users\Lindsey Wang\AppData\Local\Temp\ipykernel_2312\2221768622.py:10: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  fed
C:\Users\Lindsey Wang\AppData\Local\Temp\ipykernel_2312\2221768622.py:18: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  cpi
C:\Users\Lindsey Wang\AppData\Local\Temp\ipykernel_2312\2221768622.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  hpi
C:\Users\Lindsey Wang\AppData\Local\Temp\ipykernel_2312\2221768622.py:34: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  dpi


In [6]:
gdp["observation_date"] = gdp["observation_date"] + pd.offsets.QuarterEnd(0)

In [7]:
macro = (
    gdp
    .merge(unemp_q, on="observation_date", how="inner")
    .merge(fed_q, on="observation_date", how="inner")
    .merge(cpi_q, on="observation_date", how="inner")
    .merge(hpi_q, on="observation_date", how="inner")
    .merge(dpi_q, on="observation_date", how="inner")
)

In [8]:
macro["Year"] = macro["observation_date"].dt.year
macro["Quarter"] = macro["observation_date"].dt.quarter

macro = macro[
    [
        "observation_date",
        "Year",
        "Quarter",
        "Real_GDP",
        "Unemployment_Rate",
        "Federal_Funds_Rate",
        "CPI",
        "House_Price_Index",
        "Disposable_Personal_Income",
    ]
]

In [9]:
macro["GDP_Growth"] = macro["Real_GDP"].pct_change(1) * 100

macro["CPI_Rate"] = macro["CPI"].pct_change(1) * 100

macro["HPI_Growth"] = macro["House_Price_Index"].pct_change(1) * 100

macro["Income_Growth"] = macro["Disposable_Personal_Income"].pct_change(1) * 100

In [10]:
print(macro.head())

print(macro.info())

print(macro.describe())

  observation_date  Year  Quarter   Real_GDP  Unemployment_Rate  \
0       2005-03-31  2005        1  15844.727           5.300000   
1       2005-06-30  2005        2  15922.782           5.100000   
2       2005-09-30  2005        3  16047.587           4.966667   
3       2005-12-31  2005        4  16136.734           4.966667   
4       2006-03-31  2006        1  16353.835           4.733333   

   Federal_Funds_Rate         CPI  House_Price_Index  \
0            2.470000  192.366667         163.482333   
1            2.943333  193.666667         169.349333   
2            3.460000  196.600000         174.580667   
3            3.980000  198.433333         179.541000   
4            4.456667  199.466667         183.324000   

   Disposable_Personal_Income  GDP_Growth  CPI_Rate  HPI_Growth  Income_Growth  
0                11241.433333         NaN       NaN         NaN            NaN  
1                11344.133333    0.492624  0.675793    3.588767       0.913585  
2                

In [11]:
macro.to_csv("../outputs/Master_Macroeconomic_Data_Quarterly.csv", index=False)